In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os
from math import ceil
from scipy.stats import gaussian_kde
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import random

from scipy.linalg import orthogonal_procrustes
from scipy.linalg import solve
from scipy.linalg import orthogonal_procrustes


try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

In [2]:
cu2mci_proteomics = pd.read_csv("../data/GNPC_Harmonized_Dataset_V1/cu2mci_proteomics_data.csv")
cu2mci_proteomics.head()

,person_id,sample_id,sex,contributor_code,sample_type,visit,age_at_visit,ad,ftd,pd,...,seq_9984_12,seq_9986_14,seq_9987_30,seq_9989_12,seq_9991_112,seq_9993_11,seq_9994_217,seq_9995_6,seq_9997_12,seq_9999_1
0,f9e2f741-a44a-4ce2-9a83-b4a9028f6427,a99cd012-ae2d-4d3d-9b1e-5c8f588c6396,1,B,Sample,2,64,0,0,0,...,566.1,3653.5,550.3,400.1,834.7,1145.3,1531.4,8447.4,37763.3,3512.4
1,426976f1-388f-4fd9-b39c-b582912b108f,f12137ff-9807-4ed4-b071-c55da84318d7,2,B,Sample,2,65,0,0,0,...,581.9,1034.0,534.9,347.9,459.4,1700.4,1440.7,11518.2,29796.5,5313.6
2,9d0094df-3bc2-4837-989d-6028720d5a66,9d0094df-3bc2-4837-989d-6028720d5a66,2,I,Sample,1,63,-1,-1,-1,...,539.3,5499.5,508.9,411.9,474.6,950.0,1280.6,4655.7,37296.3,4156.4
3,d4d11a95-e9b0-4408-9bf9-cf67bef3de83,d4d11a95-e9b0-4408-9bf9-cf67bef3de83,1,B,Sample,1,50,0,0,0,...,614.7,3337.4,658.4,413.7,559.4,1200.4,1437.3,1986.8,11742.5,1284.7
4,c64c810f-7e38-4105-89a2-3d5c1de98e0e,c64c810f-7e38-4105-89a2-3d5c1de98e0e,2,B,Sample,1,65,0,0,0,...,617.3,2094.7,596.1,407.8,697.9,1105.7,1359.2,2022.5,16409.8,2566.4


In [3]:
proteomics_cols = [c for c in cu2mci_proteomics.columns if c.startswith("seq_")]

cols_with_nan_or_minus1 = [
    c for c in proteomics_cols
    if cu2mci_proteomics[c].isna().any() or (cu2mci_proteomics[c] == -1).any()
]
cu2mci_proteomics_clean = cu2mci_proteomics.drop(columns=cols_with_nan_or_minus1)

In [4]:
patient_dict = {
    pid: pdf.sort_values("visit")
    for pid, pdf in cu2mci_proteomics_clean.groupby("person_id")
}


In [5]:
clean_proteomics_cols = [c for c in cu2mci_proteomics_clean.columns if c.startswith("seq_")]

clean_cols_with_nan_or_minus1 = [
    c for c in clean_proteomics_cols
    if cu2mci_proteomics_clean[c].isna().any() or (cu2mci_proteomics_clean[c] == -1).any()
]
print("Number of proteomics cols with missing value after cleaning:", len(clean_cols_with_nan_or_minus1))

Number of proteomics cols with missing value after cleaning: 0


In [6]:
visits_group = {}
for pid, visits in patient_dict.items():
    num_visits = len(visits)
    if num_visits not in visits_group:
        visits_group[num_visits] = []
    visits_group[num_visits].append(pid)
for num_visits, pids in visits_group.items():
    print(f"Number of visits: {num_visits}, Number of patients: {len(pids)}")

Number of visits: 2, Number of patients: 48
Number of visits: 3, Number of patients: 134
Number of visits: 4, Number of patients: 24
Number of visits: 5, Number of patients: 8


In [7]:
patient_data = {}
for pid, visits in patient_dict.items():
    patient_data[pid] = {"ages": visits["age_at_visit"].to_numpy()}
    proteomics_cols = [c for c in visits.columns if c.startswith("seq_")]
    patient_data[pid]["proteomics"] = visits[proteomics_cols].to_numpy()


In [8]:
for pid, data in patient_data.items():
    print(f"Patient ID: {pid}, Ages: {data['ages']}, Proteomics shape: {data['proteomics'].shape}")

Patient ID: 0026f18a-c3f6-4255-a173-14ab2981006e, Ages: [62 63], Proteomics shape: (2, 7596)
Patient ID: 003e3031-8478-4f29-af97-dec0815ca37e, Ages: [59 62], Proteomics shape: (2, 7596)
Patient ID: 03389ff6-bf0d-4c2e-b789-a940477a18da, Ages: [62 66 69], Proteomics shape: (3, 7596)
Patient ID: 05efe196-294b-44b3-bb34-77aeabc4297c, Ages: [75 77 80 81], Proteomics shape: (4, 7596)
Patient ID: 060dbe34-3c5b-4d77-a753-2a6c65df281a, Ages: [50 54 58], Proteomics shape: (3, 7596)
Patient ID: 06204f24-20be-4939-a8e1-2c9a6d6b4239, Ages: [80 82 86], Proteomics shape: (3, 7596)
Patient ID: 08ff0521-012a-4132-842d-239f13eb4d5f, Ages: [59 64 67], Proteomics shape: (3, 7596)
Patient ID: 09e0ec3b-e74a-4f3a-971a-05c5e7b31906, Ages: [64 67 70], Proteomics shape: (3, 7596)
Patient ID: 0c374d15-669f-44d5-9c2c-298dbf2469cf, Ages: [63 66 71], Proteomics shape: (3, 7596)
Patient ID: 0f55b2f8-a793-4dfc-9b7d-da1a71731215, Ages: [79 82 85 87], Proteomics shape: (4, 7596)
Patient ID: 10a8d365-724f-4fe2-a7b8-cc5c

In [9]:
import time
from functools import wraps

def timed(name=None, verbose=True):
    """
    Simple timing decorator.
    """
    def decorator(func):
        label = name if name is not None else func.__name__

        @wraps(func)
        def wrapper(*args, **kwargs):
            t0 = time.perf_counter()
            out = func(*args, **kwargs)
            t1 = time.perf_counter()
            dt = t1 - t0
            if verbose:
                print(f"[TIMER] {label:<25s}: {dt:8.3f} s")
            return out
        return wrapper
    return decorator


In [10]:
@timed("data_loading")
def data_loading(patient_data):
    dts_all = []
    Ys = []
    Ps = []
    ts = []

    for pid, data in patient_data.items():
        ages = data['ages']
        proteomics = data['proteomics']

        Ys.append(np.log2(proteomics + 1e-6))
        ts.append(ages)
        dt = np.diff(ages)  # irregular dt
        dts_all.append(dt)
        Ps.append(pid)

    return Ys, dts_all, ts, Ps


In [11]:
y_list, _, t_list, _ = data_loading(patient_data)

Nsub = len(y_list)
print(f"Number of subjects: {Nsub}")
K = 10
print(f"Number of factors: {K}")
D = y_list[0].shape[1]
print(f"Number of features: {D}")


[TIMER] data_loading             :    0.044 s
Number of subjects: 214
Number of factors: 10
Number of features: 7596


In [12]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(123)
np.random.seed(123)
# Convert data to torch
y_torch = [torch.tensor(y, dtype=torch.float32) for y in y_list]
t_torch = [torch.tensor(t, dtype=torch.float32) for t in t_list]

In [13]:
import numpy as np
import matplotlib.pyplot as plt
import torch

def estimate_latent_trajectory(model, i, t_torch_i):
    """
    Deterministic latent propagation for one subject and one model
    """
    x_est = []
    x_prev = model.x0[i]

    x_est.append(x_prev)

    for n in range(1, len(t_torch_i)):
        dt = t_torch_i[n] - t_torch_i[n - 1]
        x_next = model.propagate_latent(x_prev, dt)
        x_est.append(x_next)
        x_prev = x_next

    return torch.stack(x_est, dim=0)  # (T_i, K)


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim


class DeterministicLatentODE_NMF(nn.Module):
    def __init__(self, K, D, Nsub):
        super().__init__()

        self.K = K
        self.D = D
        self.Nsub = Nsub

        # -----------------------------
        # Latent dynamics parameters
        # -----------------------------
        self.theta = nn.Parameter(torch.ones(K) * 0.3)
        self.mu0   = nn.Parameter(torch.zeros(K))
        self.x0    = nn.Parameter(torch.randn(Nsub, K))

        # -----------------------------
        # NMF-style nonnegative Lambda
        # -----------------------------
        self.Lambda_raw = nn.Parameter(torch.randn(D, K))
        self.log_sigma_obs = nn.Parameter(torch.tensor(-1.0))

    def Lambda(self):
        # smooth, strictly positive
        return torch.nn.functional.softplus(self.Lambda_raw)

    def propagate_latent(self, x_prev, dt):
        phi = torch.exp(-self.theta * dt)
        x_next = self.mu0 + phi * (x_prev - self.mu0)
        return x_next

    def forward(self, y_list, t_list):
        sigma_obs2 = torch.exp(self.log_sigma_obs)
        Lambda = self.Lambda()

        total_nll = 0.0

        for i, (y_i, t_i) in enumerate(zip(y_list, t_list)):
            T_i = y_i.shape[0]

            x_pred = []
            x_prev = self.x0[i]
            x_pred.append(x_prev)

            for n in range(1, T_i):
                dt = t_i[n] - t_i[n - 1]
                x_next = self.propagate_latent(x_prev, dt)
                x_pred.append(x_next)
                x_prev = x_next

            x_pred = torch.stack(x_pred, dim=0)      # (T_i, K)
            y_hat = x_pred @ Lambda.T               # (T_i, D)

            resid = y_i - y_hat
            total_nll += (
                resid.pow(2).sum() / (2 * sigma_obs2)
                + 0.5 * y_i.numel() * torch.log(sigma_obs2)
            )

        # Regularization (important for identifiability)
        total_nll += 1e-3 * self.x0.pow(2).sum()
        total_nll += 1e-3 * Lambda.pow(2).sum()

        return total_nll



In [ ]:
@timed(name="train_one_run")
def train_one_run(y_torch, t_torch, K, D, Nsub,
                  lr=1e-2, n_epochs=3000, seed=None):

    if seed is not None:
        torch.manual_seed(seed)

    model = DeterministicLatentODE_NMF(K=K, D=D, Nsub=Nsub)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    loss_history = []

    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = model(y_torch, t_torch)
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

    return model, np.array(loss_history)


In [16]:
n_runs = 10

models = []
loss_curves = []

for r in range(n_runs):
    model_r, loss_r = train_one_run(
        y_torch, t_torch,
        K=K, D=D, Nsub=Nsub,
        seed=100 + r
    )
    models.append(model_r)
    loss_curves.append(loss_r)

loss_curves = np.stack(loss_curves)  # (n_runs, n_epochs)


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

mean_loss = loss_curves.mean(axis=0)
std_loss = loss_curves.std(axis=0)

plt.figure(figsize=(6,4))
plt.plot(mean_loss, label="Mean NLL")
plt.fill_between(
    np.arange(len(mean_loss)),
    mean_loss - 1.96 * std_loss,
    mean_loss + 1.96 * std_loss,
    alpha=0.3,
    label="95% CI"
)
plt.xlabel("Epoch")
plt.ylabel("Negative log-likelihood")
plt.legend()
plt.title("Training loss across runs")
plt.tight_layout()
plt.show()


In [ ]:
with torch.no_grad():
    for i in range(Nsub):
        T_i = len(t_list[i])
        t_i = t_list[i]

        # Collect trajectories from all runs
        x_est_runs = []

        for model in models:
            x_est_i = estimate_latent_trajectory(
                model,
                i,
                t_torch[i]
            )
            x_est_runs.append(x_est_i)

        # Shape: (n_runs, T_i, K)
        x_est_runs = torch.stack(x_est_runs).cpu().numpy()

        # Mean and variance
        x_mean = x_est_runs.mean(axis=0)
        x_std  = x_est_runs.std(axis=0)

        x_true = y_list[i] @ torch.linalg.pinv(model.Lambda().cpu()).numpy()

        fig, axes = plt.subplots(K, 1, figsize=(6, 4 * K), sharex=True)

        if K == 1:
            axes = [axes]

        for k in range(K):
            # True latent (OU)
            axes[k].plot(
                t_i,
                x_true[:, k],
                'k-',
                lw=2,
                label="True (OU)"
            )

            # Mean estimate
            axes[k].plot(
                t_i,
                x_mean[:, k],
                'b--',
                lw=2,
                label="ODE fit (mean)"
            )

            # Variance / CI shading
            axes[k].fill_between(
                t_i,
                x_mean[:, k] - x_std[:, k],
                x_mean[:, k] + x_std[:, k],
                color='blue',
                alpha=0.3,
                label="±1 std" if k == 0 else None
            )

            axes[k].set_title(
                f"Subject {i+1}, latent dim {k+1}"
            )
            axes[k].legend()

        axes[-1].set_xlabel("Time")
        plt.tight_layout()
        plt.show()

